# Gold: Price by temperature band
This notebook reads the joined Silver dataset, filters out hours without a temperature measurement, groups electricity prices into 5°C temperature bands, and merges the summary statistics into `gold.price_by_temperature_band`.

### Key design choices in this notebook
* **Explicit filtering of missing temperatures:** Because Silver preserves `NULL` weather values without imputation, this notebook explicitly filters on `air_temperature_c.isNotNull()` so only hours with actual measured temperatures are assigned to a band.
* **Integer sort order (`temperature_band_order`):** Text labels like `"-10 to -5 °C"` and `"-5 to 0 °C"` sort incorrectly in alphabetical order. Computing `floor(air_temperature_c / 5.0)` as an integer column (`temperature_band_order`) gives dashboards and downstream queries a numeric key to sort bands from coldest to warmest.
* **Both mean and median electricity price:** Electricity prices often have right-skewed spikes during cold hours. Calculating both `avg(nok_per_kwh)` and `percentile_approx(nok_per_kwh, 0.5)` shows both the average cost and a typical hour inside each temperature band.
* **Upsert with scoped delete (`whenNotMatchedBySourceDelete`):** Since Gold is a full aggregation across `silver.weather_price_hourly`, the Delta `MERGE` updates existing bands, inserts new ones, and deletes any stale `NO2` bands that no longer exist in the source data.

In [0]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Lock Spark session to UTC and dynamically resolve the active Unity Catalog
spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = spark.sql("SELECT current_catalog()").first()[0]

# Read run_id from the job widget (or generate a new UUID for interactive runs)
dbutils.widgets.text("run_id", "")
RUN_ID = dbutils.widgets.get("run_id") or str(uuid.uuid4())

### Load joined Silver data with valid temperature readings
Reads `silver.weather_price_hourly` for `NO2` and excludes any hours where `air_temperature_c` is `NULL`. Fails immediately if no valid rows are found.

In [0]:
# Load joined Silver data for NO2, keeping only hours with a measured temperature
source_df = (
    spark.table(f"{CATALOG}.silver.weather_price_hourly")
    .filter(
        (F.col("price_area") == "NO2")
        & F.col("air_temperature_c").isNotNull()
    )
)

# Fail fast if there are no rows to aggregate
if source_df.count() == 0:
    raise ValueError("Gold source is empty")

### Aggregate price and wind metrics by 5°C temperature band
Assigns each hour to a 5°C bucket using `floor(air_temperature_c / 5.0)`, builds the human-readable band label (such as `"-5 to 0 °C"`), and computes observation count, mean, median, min, max price, and average wind speed.

In [0]:
# Group hourly data into 5°C temperature bands and calculate price and wind statistics
gold_df = (
    source_df
    # Integer bucket index (e.g. -2 for -10 to -5 °C, -1 for -5 to 0 °C, 0 for 0 to 5 °C) for sorting
    .withColumn(
        "temperature_band_order",
        F.floor(F.col("air_temperature_c") / F.lit(5.0)).cast("int")
    )
    # Human-readable label for charts and tables
    .withColumn(
        "temperature_band",
        F.concat(
            (F.col("temperature_band_order") * 5).cast("string"),
            F.lit(" to "),
            ((F.col("temperature_band_order") * 5) + 5).cast("string"),
            F.lit(" °C")
        )
    )
    .groupBy(
        "price_area", "temperature_band_order", "temperature_band"
    )
    # Compute count, mean, median (percentile_approx 0.5), min, max price, and mean wind speed
    .agg(
        F.count("*").alias("observation_count"),
        F.round(F.avg("nok_per_kwh"), 4)
        .alias("average_price_nok_per_kwh"),
        F.round(
            F.expr("percentile_approx(nok_per_kwh, 0.5)"), 4
        ).alias("median_price_nok_per_kwh"),
        F.round(F.min("nok_per_kwh"), 4)
        .alias("minimum_price_nok_per_kwh"),
        F.round(F.max("nok_per_kwh"), 4)
        .alias("maximum_price_nok_per_kwh"),
        F.round(F.avg("wind_speed_mps"), 2)
        .alias("average_wind_speed_mps")
    )
    .withColumn("updated_at_utc", F.current_timestamp())
)

if gold_df.count() == 0:
    raise ValueError("Gold output is empty")

### Merge into Gold table and log data quality check
Merges the aggregated bands into `gold.price_by_temperature_band` on `(price_area, temperature_band_order)`, removes any outdated `NO2` bands via `whenNotMatchedBySourceDelete`, and logs the `gold_output_nonempty` check to `ops.data_quality_results`.


In [0]:
# Merge temperature bands into Gold and remove any stale bands for NO2
(
    DeltaTable.forName(
        spark, f"{CATALOG}.gold.price_by_temperature_band"
    ).alias("target")
    .merge(
        gold_df.alias("source"),
        """
        target.price_area = source.price_area
        AND target.temperature_band_order = source.temperature_band_order
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete("target.price_area = 'NO2'")
    .execute()
)

# Log an ERROR check confirming that the Gold table contains aggregated rows
check = [(
    RUN_ID, "gold", "gold_output_nonempty", "ERROR", True,
    float(gold_df.count()), "Gold contains temperature-band rows",
    datetime.now(timezone.utc).replace(tzinfo=None)
)]

dq_df = spark.createDataFrame(check, """
    run_id string, layer string, check_name string, severity string,
    passed boolean, actual_value double, message string,
    checked_at_utc timestamp
""")

# Upsert the check result into ops.data_quality_results
(
    DeltaTable.forName(
        spark, f"{CATALOG}.ops.data_quality_results"
    ).alias("target")
    .merge(
        dq_df.alias("source"),
        """
        target.run_id = source.run_id
        AND target.layer = source.layer
        AND target.check_name = source.check_name
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"Gold analysis completed. run_id={RUN_ID}")
display(gold_df.orderBy("temperature_band_order"))